In [15]:
import torch
from torch_geometric.data import Data, Dataset
from torch_geometric.data.data import DataEdgeAttr, DataTensorAttr, GlobalStorage

from pathlib import Path
import math
from pyproj import Transformer
from sumolib.net import readNet

In [4]:
def calculate_edge_length(shape):
    length = 0.0
    # shape = {(x0, y0), (x1, y1), ... , (xn, yn)} --> lista di tuple
    # iterative eucledian distance computation to get the total lenght
    for i in range(1, len(shape)):
        x1, y1 = shape[i - 1]
        x2, y2 = shape[i]
        length += math.sqrt((x1 - x2) ** 2 + (y1 - y2) ** 2)
    return length

In [11]:
net_file = "/home/marta/tesi-5t/sumo-stgcn/input/francia_peschiera_passenger.net.xml"
net = readNet(
        net_file, withPrograms=True, withConnections=True, withInternal=True
    )

edge_id_to_idx = {}
edge_static_features = []

for idx, edge in enumerate(net.getEdges()):
    if edge.getID().startswith(":"):
        continue
    edge_id_to_idx[edge.getID()] = idx
    # edge attributes
    lanes = edge.getLanes()
    length = calculate_edge_length(edge.getShape())
    speed_limit = edge.getSpeed()
    num_lanes = len(lanes)
    priority = edge.getPriority()
    capacity = ...
    
    # tensor index = edge id - idx transformation
    # tensor elements = edge attributes
    edge_static_features.append([length, speed_limit, float(num_lanes), float(priority)])
    x_static = torch.tensor(edge_static_features, dtype=torch.long) # shape = torch.tensor
    
x_static

tensor([[ 18,   5,   1,   1],
        [ 23,   5,   1,   1],
        [ 12,   5,   1,   1],
        ...,
        [ 87,  13,   1,   3],
        [143,  13,   1,   3],
        [ 66,  13,   1,   3]])

In [ ]:
JUNCTION_TYPE_MAP = {
    "traffic_light": 0,
    "priority": 1,
    "right_before_left": 2,
    "unregulated": 3,
    "dead_end": 4,
    "unknown": 5,
    "internal": 6,
}

In [16]:
TRANSFORMER = Transformer.from_crs("EPSG:32632", "EPSG:4326", always_xy=True)

def _transform_coord(x: float, y:float):
    return TRANSFORMER.transform(x, y)

In [ ]:
src_list, dst_list, attr_list = [], [], []
seen = set()

for edge in net.getEdges():
    if edge.getID().startswith(":"):
        continue
    for lane in edge.getLanes():
        for conn in lane.getOutgoing():
            to_edge_id = conn.getToLane().getEdge().getID()
            if to_edge_id.startswith(":"):
                continue
            src_idx = edge_id_to_idx.get(edge.getID())
            dst_idx = edge_id_to_idx.get(to_edge_id)
            if src_idx is None or dst_idx is None:
                continue

            pair = (src_idx, dst_idx)
            if pair in seen:
                continue      # evita duplicati (più lane → stessa coppia edge)
            seen.add(pair)
            
            

In [ ]:
source_edges, dest_edges, attribute_list = [], [], []
seen_conn = set()

for node in net.getNodes():
    incoming = [e for e in node.getIncoming() if not e.getID().startswith(":")]
    outgoing = [e for e in node.getOutgoing() if not e.getID().startswith(":")]
    
    # print(incoming)
    jtype = JUNCTION_TYPE_MAP.get(node.getType(), 5)
    lon, lat = _transform_coord(node.getCoord()[0], node.getCoord()[1])
    has_tls = float(node.getType == "traffic_light")
    
    for e_in in incoming:
        for e_out in outgoing:
            source_id = edge_id_to_idx.get(e_in.getID())
            dest_id = edge_id_to_idx.get(e_out.getID())
            if source_id is None or dest_id is None or source_id == dest_id:
                continue
            connection_pair = (source_id, dest_id)
            if connection_pair in seen_conn:
                continue
            else:
                seen_conn.add(connection_pair)
                source_edges.append(source_id)
                dest_edges.append(dest_id)
                attribute_list.append([float(jtype), lon, lat, has_tls])
                
connections_idx = torch.tensor([source_edges, dest_edges], dtype= torch.long)

connections_idx # shape = edges x 2 (in - out)

tensor([[4933, 6667, 4930,  ..., 6764, 4902, 4903],
        [6668, 6668, 4931,  ..., 4901, 6765, 6766]])

In [22]:
edge_attr = torch.tensor(attribute_list, dtype=torch.float)
edge_attr

tensor([[ 1.0000,  7.6161, 45.0663,  0.0000],
        [ 1.0000,  7.6161, 45.0663,  0.0000],
        [ 1.0000,  7.6161, 45.0661,  0.0000],
        ...,
        [ 4.0000,  7.6146, 45.0778,  0.0000],
        [ 4.0000,  7.6146, 45.0786,  0.0000],
        [ 4.0000,  7.6165, 45.0755,  0.0000]])

In [ ]:
def build_dynamic(self, net, tls_states: dict) -> tuple:
    """
    Versione con archi filtrati per stato semaforo.
    tls_states: { tls_id: { link_index: bool } }  True=verde, False=rosso
    
    Puoi popolarlo da TraCI:
        traci.trafficlight.getRedYellowGreenState(tls_id)
    """
    edge_id_to_idx = {
        e.getID(): i for i, e in enumerate(net.getEdges())
        if not e.getID().startswith(":")
    }

    src_list, dst_list, attr_list = [], [], []
    seen = set()

    for edge in net.getEdges():
        if edge.getID().startswith(":"):
            continue
        for lane in edge.getLanes():
            for conn in lane.getOutgoing():
                to_edge_id = conn.getToLane().getEdge().getID()
                if to_edge_id.startswith(":"):
                    continue

                # Filtra per semaforo
                tls_id      = conn.getTLSID()
                tl_link_idx = conn.getTLLinkIndex()
                if tls_id:
                    phase_states = tls_states.get(tls_id, {})
                    if not phase_states.get(tl_link_idx, True):
                        continue   # rosso → arco rimosso

                src_idx = edge_id_to_idx.get(edge.getID())
                dst_idx = edge_id_to_idx.get(to_edge_id)
                if src_idx is None or dst_idx is None or (src_idx, dst_idx) in seen:
                    continue
                seen.add((src_idx, dst_idx))

                via_node  = edge.getToNode()
                jtype_enc = self.JUNCTION_TYPE_MAP.get(via_node.getType(), 5)
                x, y      = via_node.getCoord()
                lon, lat  = self._transform_coord(x, y)

                src_list.append(src_idx)
                dst_list.append(dst_idx)
                attr_list.append([float(jtype_enc), lon, lat,
                                    1.0, float(tl_link_idx if tls_id else -1)])

In [ ]:
torch.load()